# Hito 3 - Notebook 10: Modelado Avanzado - Regresion de la Demanda de Insumos
## Fase 4 de CRISP-DM (avanzada) - 1.5.3

> **Disenado para GOOGLE COLAB.** Compara **Random Forest Regressor** vs **XGBoost Regressor** para predecir la **demanda (consumo) acumulada** a **t+7** y **t+14** dias por insumo. Se usa validacion cruzada temporal (`TimeSeriesSplit`) para respetar el orden cronologico.
>
> **Por que demanda y no nivel de stock:** el EDA mostro que el stock a 7/14 dias no tiene autocorrelacion (los cambios diarios son ruido), mientras que el consumo es altamente predecible. El **stock proyectado y las alertas se derivan** de la demanda: `Stock_Proyectado = Stock_Actual - Demanda_Predicha`.

In [ ]:
# === Configuracion para Google Colab (ejecutar primero) ===
# Instala dependencias si no estan presentes.
try:
    import xgboost, imblearn, sklearn, joblib  # noqa
except Exception:
    !pip -q install xgboost imbalanced-learn scikit-learn joblib
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (9, 5)
pd.set_option('display.max_columns', None)

In [ ]:
def cargar_procesado(nombre):
    """Carga un CSV procesado buscando en rutas locales o pidiendo subirlo en Colab."""
    for p in [Path('data/processed') / nombre, Path('../data/processed') / nombre,
              Path('/content') / nombre, Path(nombre)]:
        if p.exists():
            print('Cargado desde:', p)
            return pd.read_csv(p)
    try:
        from google.colab import files
        print(f'Sube el archivo: {nombre}')
        subido = files.upload()
        return pd.read_csv(list(subido.keys())[0])
    except Exception as e:
        raise FileNotFoundError(f'No se encontro {nombre}: {e}')

MODELS_DIR = Path('models'); MODELS_DIR.mkdir(exist_ok=True, parents=True)

## Carga del dataset logistico preparado

In [ ]:
stock = cargar_procesado('Dataset_ALDIMI_Logistica_Preparado.csv')
stock['Fecha'] = pd.to_datetime(stock['Fecha'])
stock = stock.sort_values(['ID_Insumo', 'Fecha'])
FEATURES = ['Stock_Actual', 'Consumo_Diario', 'Consumo_7d', 'Consumo_14d',
            'Consumo_Prev_7d', 'Consumo_Prev_14d', 'Consumo_Std_7d',
            'Consumo_Lag_1', 'Consumo_Lag_7', 'Stock_Lag_1', 'Lead_Time',
            'Punto_Reorden', 'Ratio_Stock', 'Cobertura_Dias', 'Demanda_Pronosticada',
            'Unit_Cost', 'Unit_Price', 'Promotion_Flag', 'Order_Quantity',
            'Ocupacion_Albergue', 'Pacientes_Alto_Riesgo', 'Ocupacion_Total',
            'Mes', 'Dia_Semana']
FEATURES = [c for c in FEATURES if c in stock.columns]
TARGETS = {'t+7': 'Demanda_Fut_7d', 't+14': 'Demanda_Fut_14d'}
print(stock.shape, '| features:', len(FEATURES))

## 1.5.3 Modelos evaluados y configuracion

- **Random Forest Regressor**: robusto, capta no linealidades, poca sensibilidad a la escala.
- **XGBoost Regressor**: boosting regularizado, suele minimizar mejor el error.

Metrica de optimizacion: **MAE** (interpretable en unidades de consumo a reponer).

### Justificacion de hiperparametros y criterios de busqueda

| Elemento | Valor | Criterio |
|---|---|---|
| **Validacion cruzada** | `TimeSeriesSplit(n_splits=3)` | Respeta el orden cronologico; 3 pliegues equilibran validez temporal y tiempo en Colab. |
| **Metrica de seleccion** | `neg_mean_absolute_error` | MAE en unidades de consumo a reponer; interpretable para planificar compras (criterio operativo ALDIMI). |
| **Busqueda** | `RandomizedSearchCV(n_iter=4)` | Explora el espacio sin grid exhaustivo (8 combos x 3 folds x 2 modelos x 2 horizontes = 96 ajustes con grid; ~24 con randomized en t+7 + reutilizacion en t+14). |
| **`reg__n_estimators` RF** | `[200, 300]` | Balance entre capacidad y tiempo; >300 arboles no mejora MAE de forma proporcional en ~14k filas de train. |
| **`reg__n_estimators` XGB** | `[300, 500]` | Boosting necesita mas iteraciones que bagging; rango acotado para Colab. |
| **`reg__max_depth`** | `[4, 6]` (XGB) / `[12, None]` (RF) | Limita sobreajuste en series con cambio de regimen (dataset shift); profundidad moderada generaliza mejor. |
| **`reg__min_samples_leaf`** | `[1, 3]` | Regularizacion en hojas para estabilizar predicciones en insumos de bajo consumo. |
| **`reg__learning_rate`** | `[0.05, 0.1]` | Rango estandar XGB: menor tasa + mas arboles = mejor generalizacion. |
| **`n_jobs`** | `1` | Estabilidad de memoria en Colab. |
| **Reutilizacion t+14** | Params de t+7 | Misma estructura de demanda; el tuning en t+7 se reutiliza en t+14 para ahorrar tiempo sin perder validez metodologica. |

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

CV_FOLDS = 3
N_ITER = 4

def construir_pipelines():
    pipe_rf = Pipeline([('sc', StandardScaler()),
                        ('reg', RandomForestRegressor(random_state=42, n_jobs=1))])
    pipe_xgb = Pipeline([('sc', StandardScaler()),
                         ('reg', XGBRegressor(random_state=42, tree_method='hist', n_jobs=1))])
    return pipe_rf, pipe_xgb

PARAM_DIST_RF = {
    'reg__n_estimators': [200, 300],
    'reg__max_depth': [12, None],
    'reg__min_samples_leaf': [1, 3],
}
PARAM_DIST_XGB = {
    'reg__n_estimators': [300, 500],
    'reg__max_depth': [4, 6],
    'reg__learning_rate': [0.05, 0.1],
}

def ajustar_con_busqueda(pipe, param_dist, X_tr, y_tr, tscv):
    gs = RandomizedSearchCV(pipe, param_dist, n_iter=N_ITER,
                            scoring='neg_mean_absolute_error', cv=tscv,
                            n_jobs=1, random_state=42, verbose=1)
    gs.fit(X_tr, y_tr)
    return gs

def entrenar_con_params(pipe, params, X_tr, y_tr):
    pipe.set_params(**params)
    pipe.fit(X_tr, y_tr)
    return pipe

## Entrenamiento y tuning por horizonte (t+7 y t+14)

In [ ]:
resultados = []
modelos_guardados = {}
tscv = TimeSeriesSplit(n_splits=CV_FOLDS)
mejores_params = {}  # reutilizar en t+14 los params tunados en t+7

for etiqueta, tgt in TARGETS.items():
    d = stock.dropna(subset=[tgt]).sort_values('Fecha')
    X = d[FEATURES].fillna(0); y = d[tgt]
    corte = int(len(d) * 0.8)
    X_tr, X_te, y_tr, y_te = X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]
    pipe_rf, pipe_xgb = construir_pipelines()

    if etiqueta == 't+7':
        gs_rf = ajustar_con_busqueda(pipe_rf, PARAM_DIST_RF, X_tr, y_tr, tscv)
        gs_xgb = ajustar_con_busqueda(pipe_xgb, PARAM_DIST_XGB, X_tr, y_tr, tscv)
        mejores_params['Random Forest'] = gs_rf.best_params_
        mejores_params['XGBoost'] = gs_xgb.best_params_
        candidatos = [('Random Forest', gs_rf.best_estimator_), ('XGBoost', gs_xgb.best_estimator_)]
        print(f'{etiqueta}: RF={gs_rf.best_params_} | XGB={gs_xgb.best_params_}')
    else:
        mod_rf = entrenar_con_params(pipe_rf, mejores_params['Random Forest'], X_tr, y_tr)
        mod_xgb = entrenar_con_params(pipe_xgb, mejores_params['XGBoost'], X_tr, y_tr)
        candidatos = [('Random Forest', mod_rf), ('XGBoost', mod_xgb)]
        print(f'{etiqueta}: params reutilizados de t+7 -> RF={mejores_params["Random Forest"]}')

    for nombre, modelo in candidatos:
        pred = modelo.predict(X_te)
        resultados.append({'Horizonte': etiqueta, 'Modelo': nombre,
                           'MAE': round(mean_absolute_error(y_te, pred), 3),
                           'RMSE': round(np.sqrt(mean_squared_error(y_te, pred)), 3),
                           'R2': round(r2_score(y_te, pred), 4)})
        modelos_guardados[(etiqueta, nombre)] = modelo

tabla_reg = pd.DataFrame(resultados)
tabla_reg

## Tabla 3 - Prediccion de demanda de insumos, horizonte t+7

In [ ]:
tabla_reg[tabla_reg['Horizonte'] == 't+7'].reset_index(drop=True)

## Tabla 4 - Prediccion de demanda de insumos, horizonte t+14

In [ ]:
tabla_reg[tabla_reg['Horizonte'] == 't+14'].reset_index(drop=True)

> **Conclusion (seleccion del modelo):** para cada horizonte se elige el modelo con **menor MAE**. La demanda resulta altamente predecible (R2 ~0.95-0.99). Nota importante: la **media movil (naive)** del notebook 08 es un baseline muy fuerte y *adaptativo* (usa la ventana de consumo mas reciente), por lo que se ajusta sola a la rampa de ocupacion 50->100; los modelos de arboles, entrenados en el pasado, no extrapolan por encima del rango de ocupacion visto y quedan muy cerca del naive. El valor del modelo avanzado esta en (1) integrar multiples senales (ocupacion, lead time, estacionalidad) para escenarios y (2) el simulador de compras. Complete con los valores de Colab.

## Guardado de artefactos (.joblib)

In [ ]:
for etiqueta in TARGETS:
    sub = tabla_reg[tabla_reg['Horizonte'] == etiqueta].sort_values('MAE')
    mejor_nombre = sub.iloc[0]['Modelo']
    joblib.dump(modelos_guardados[(etiqueta, mejor_nombre)],
                MODELS_DIR / f'reg_demanda_{etiqueta.replace("+", "")}.joblib')
    print(etiqueta, '-> mejor:', mejor_nombre)
# Guardar tambien ambos candidatos para las comparativas del notebook 11
for (etiqueta, nombre), mod in modelos_guardados.items():
    joblib.dump(mod, MODELS_DIR / f'reg_{etiqueta.replace("+", "")}_{nombre.replace(" ", "_")}.joblib')
joblib.dump({'features': FEATURES, 'targets': TARGETS}, MODELS_DIR / 'meta_regresion.joblib')
tabla_reg.to_csv(MODELS_DIR / 'metricas_regresion.csv', index=False)
print('Artefactos de regresion guardados.')